In [1]:
"""
PAINT Dataset Loader - Complete & Concise
==========================================
Loads and reorganizes heliostat data into a clean structure:

paint_data/
└── AA31/
    ├── Calibration/
    │   ├── 118385/
    │   │   ├── flux.png
    │   │   └── properties.json
    │   ├── 118423/
    │   │   ├── flux.png
    │   │   └── properties.json
    │   └── ...
    ├── Deflectometry/
    │   └── *.csv
    └── Properties/
        └── *.json
"""

import pathlib
import shutil
import torch
import pandas as pd
import numpy as np
import json
from torch.utils.data import Dataset, DataLoader, random_split
import paint.util.paint_mappings as mappings
from paint.data.dataset import PaintCalibrationDataset
from paint.data import StacClient
from paint.util import set_logger_config

set_logger_config()

In [ ]:
# ============================================================================
# CELL 1: Download & Reorganize Data
# ============================================================================

def download_and_organize_data(heliostat_id, data_root="./paint_data"):
    """Download all data and reorganize into clean structure."""
    data_root = pathlib.Path(data_root)
    heliostat_dir = data_root / heliostat_id
    
    print("="*80)
    print(f"Setting up data for {heliostat_id}")
    print("="*80)
    
    # Download flux images and metadata (if not already present)
    print("\n📥 Step 1: Downloading flux images and metadata...")
    flux_dataset = PaintCalibrationDataset.from_heliostats(
        heliostats=[heliostat_id],
        root_dir=heliostat_dir / "flux_images",
        item_type=mappings.CALIBRATION_FLUX_IMAGE_KEY,
        download=True,
    )
    
    metadata_dataset = PaintCalibrationDataset.from_heliostats(
        heliostats=[heliostat_id],
        root_dir=heliostat_dir/ "metadata",
        item_type=mappings.CALIBRATION_PROPERTIES_KEY,
        download=True,
    )
    
    # Download deflectometry and properties
    print("\n📥 Step 2: Downloading deflectometry and properties...")
    client = StacClient(output_dir=str(data_root))
    
    try:
        client.get_heliostat_data(heliostats=[heliostat_id], collections=['Deflectometry'])
        print("✓ Deflectometry downloaded")
    except Exception as e:
        print(f"⚠️  Deflectometry: {e}")
    
    try:
        client.get_heliostat_data(heliostats=[heliostat_id], collections=['Properties'])
        print("✓ Properties downloaded")
    except Exception as e:
        print(f"⚠️  Properties: {e}")
    
    # Reorganize into clean structure
    print("\n🔄 Step 3: Reorganizing into Calibration/ folder...")
    calibration_dir = heliostat_dir / "Calibration"
    calibration_dir.mkdir(parents=True, exist_ok=True)
    
    # Get source directories
    flux_src = heliostat_dir/ "flux_images"
    meta_src = heliostat_dir / "metadata"
    
    # Find all flux and metadata files
    flux_files = list(flux_src.glob("*-flux.png"))
    meta_files = list(meta_src.glob("*-calibration-properties.json"))
    
    # Extract calibration IDs and create mapping
    flux_map = {f.name.split('-')[0]: f for f in flux_files}
    meta_map = {f.name.split('-')[0]: f for f in meta_files}
    
    # Find matching pairs
    common_ids = set(flux_map.keys()) & set(meta_map.keys())
    
    print(f"  Found {len(common_ids)} matching calibration pairs")
    
    # Copy files into organized structure
    for cal_id in sorted(common_ids):
        cal_folder = calibration_dir / cal_id
        cal_folder.mkdir(exist_ok=True)
        
        # Copy flux image
        if not (cal_folder / "flux.png").exists():
            shutil.copy2(flux_map[cal_id], cal_folder / "flux.png")
        
        # Copy metadata
        if not (cal_folder / "properties.json").exists():
            shutil.copy2(meta_map[cal_id], cal_folder / "properties.json")
    
    print(f"✓ Organized {len(common_ids)} calibrations into {calibration_dir}")
    
    # Show final structure
    print("\n📁 Final structure:")
    print(f"  {heliostat_dir.name}/")
    for subdir in sorted(heliostat_dir.iterdir()):
        if subdir.is_dir():
            count = len(list(subdir.glob("*")))
            print(f"    ├── {subdir.name}/  ({count} items)")
    
    print("="*80)
    
    return len(common_ids)


# Run once to download and organize
num_calibrations = download_and_organize_data("AA31")



Setting up data for AA31

📥 Step 1: Downloading flux images and metadata...
[2025-12-26 00:20:26,300][paint.data.dataset][INFO] - Initializing a dataset based on heliostats. The heliostats included are:
 ['AA31']
[2025-12-26 00:20:26,300][paint.data.dataset][INFO] - Beginning to download the data...
[2025-12-26 00:20:26,301][paint.data.stac_client][INFO] - Loading catalogs for desired heliostats. 
[2025-12-26 00:20:26,424][paint.data.stac_client][INFO] - Processing heliostat catalog AA31-heliostat-catalog


Processing Items in Heliostat AA31-heliostat-catalog: 100%|██████████| 135/135 [00:01<00:00, 71.76Item/s]

[2025-12-26 00:20:47,330][paint.data.dataset][INFO] - Initializing a data set for flux_image calibration items...
[2025-12-26 00:20:47,330][paint.data.dataset][INFO] - Note that this is a dataset containing images!
[2025-12-26 00:20:47,332][paint.data.dataset][INFO] - Initializing a dataset based on heliostats. The heliostats included are:
 ['AA31']
[2025-12-26 00:20:47,332][paint.data.dataset][INFO] - Beginning to download the data...
[2025-12-26 00:20:47,333][paint.data.stac_client][INFO] - Loading catalogs for desired heliostats. 


[2025-12-26 00:20:47,471][paint.data.stac_client][INFO] - Processing heliostat catalog AA31-heliostat-catalog


Processing Items in Heliostat AA31-heliostat-catalog: 100%|██████████| 135/135 [00:02<00:00, 58.23Item/s]

...

[2025-12-26 00:21:09,112][paint.data.stac_client][INFO] - Processing heliostat catalog AA31-heliostat-catalog


Processing Items in Heliostat AA31-heliostat-catalog: 100%|██████████| 2/2 [00:13<00:00,  6.56s/Item]

✓ Deflectometry downloaded
[2025-12-26 00:21:23,175][paint.data.stac_client][INFO] - Resuming download from checkpoint!
[2025-12-26 00:21:23,322][paint.data.stac_client][INFO] - Processing heliostat catalog AA31-heliostat-catalog



Processing Items in Heliostat AA31-heliostat-catalog: 100%|██████████| 1/1 [00:00<00:00,  6.25Item/s]


✓ Properties downloaded

🔄 Step 3: Reorganizing into Calibration/ folder...
  Found 132 matching calibration pairs
✓ Organized 132 calibrations into paint_data/AA31/Calibration

📁 Final structure:
  AA31/
    ├── Calibration/  (132 items)
    ├── Deflectometry/  (6 items)
    ├── Properties/  (1 items)


In [3]:
# ============================================================================
# CELL 2: Dataset Class
# ============================================================================

class HeliostatDataset(Dataset):
    """Loads organized heliostat calibration data."""
    
    def __init__(self, heliostat_id, data_root="./paint_data", transform=None):
        self.heliostat_id = heliostat_id
        self.data_root = pathlib.Path(data_root)
        self.transform = transform
        
        # Get calibration folders
        calibration_dir = self.data_root / heliostat_id / "Calibration"
        self.calibration_folders = sorted([d for d in calibration_dir.iterdir() if d.is_dir()])
        
        # Load static data (same for all calibrations)
        self.deflectometry = self._load_csv(self.data_root / heliostat_id / "Deflectometry", "*-filled-*.csv")
        self.properties = self._load_json(self.data_root / heliostat_id / "Properties", "*.json")
        
        print(f"✓ Dataset: {len(self)} calibrations")
        print(f"  Deflectometry: {'✓' if self.deflectometry is not None else '✗'}")
        print(f"  Properties: {'✓' if self.properties else '✗'}")
    
    def _load_csv(self, directory, pattern):
        """Load CSV file as tensor."""
        if not directory.exists():
            return None
        files = list(directory.glob(pattern))
        if not files:
            files = [f for f in directory.glob("*.csv") if not f.name.startswith('.')]
        if not files:
            return None
        df = pd.read_csv(sorted(files)[-1])
        return torch.tensor(df.values, dtype=torch.float32)
    
    def _load_json(self, directory, pattern):
        """Load JSON file as dict."""
        if not directory.exists():
            return {}
        files = list(directory.glob(pattern))
        if not files:
            return {}
        with open(sorted(files)[-1], 'r') as f:
            return json.load(f)
    
    def __len__(self):
        return len(self.calibration_folders)
    
    def __getitem__(self, idx):
        """Load one calibration sample."""
        cal_folder = self.calibration_folders[idx]
        cal_id = cal_folder.name
        
        # Load flux image
        from PIL import Image
        import torchvision.transforms.functional as TF
        flux_path = cal_folder / "flux.png"
        flux_image = TF.to_tensor(Image.open(flux_path))
        
        if self.transform:
            flux_image = self.transform(flux_image)
        
        # Load metadata
        with open(cal_folder / "properties.json", 'r') as f:
            metadata = json.load(f)
        
        # Extract and convert fields
        motor_pos = metadata.get('motor_position', [0.0, 0.0])
        if isinstance(motor_pos, dict):
            motor_pos = [motor_pos.get('azimuth', 0.0), motor_pos.get('elevation', 0.0)]
        
        focal_spot = metadata.get('focal_spot', [0.0, 0.0, 0.0])
        if isinstance(focal_spot, dict):
            focal_spot = [focal_spot.get('x', 0.0), focal_spot.get('y', 0.0), focal_spot.get('z', 0.0)]
        
        return {
            'flux_image': flux_image,
            'sun_position': torch.tensor([
                metadata.get('sun_azimuth', 0.0),
                metadata.get('sun_elevation', 0.0)
            ], dtype=torch.float32),
            'motor_position': torch.tensor(motor_pos, dtype=torch.float32),
            'focal_spot': torch.tensor(focal_spot, dtype=torch.float32),
            'target_name': metadata.get('target_name', 'unknown'),
            'calibration_id': cal_id,
            'heliostat_id': self.heliostat_id,
            'deflectometry': self.deflectometry if self.deflectometry is not None else torch.tensor([]),
            'properties': self.properties,
        }


In [4]:
# ============================================================================
# CELL 3: Create DataLoaders
# ============================================================================

def create_dataloaders(heliostat_id, data_root="./paint_data", batch_size=16,
                      train_ratio=0.7, val_ratio=0.15):
    """Create train/val/test loaders."""
    dataset = HeliostatDataset(heliostat_id, data_root)
    
    # Split
    total = len(dataset)
    train_size = int(train_ratio * total)
    val_size = int(val_ratio * total)
    test_size = total - train_size - val_size
    
    train_set, val_set, test_set = random_split(
        dataset, [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    # Create loaders
    common_args = {'batch_size': batch_size, 'num_workers': 0, 'pin_memory': True}
    train_loader = DataLoader(train_set, shuffle=True, **common_args)
    val_loader = DataLoader(val_set, shuffle=False, **common_args)
    test_loader = DataLoader(test_set, shuffle=False, **common_args)
    
    print(f"\n✓ Loaders: Train={len(train_set)}, Val={len(val_set)}, Test={len(test_set)}")
    
    return train_loader, val_loader, test_loader, dataset


train_loader, val_loader, test_loader, dataset = create_dataloaders("AA31", batch_size=16)



✓ Dataset: 132 calibrations
  Deflectometry: ✗
  Properties: ✓

✓ Loaders: Train=92, Val=19, Test=21


In [5]:
# ============================================================================
# CELL 4: Test Loading
# ============================================================================

batch = next(iter(train_loader))

print("\n" + "="*80)
print("Batch Structure:")
print("="*80)

for key, value in batch.items():
    if isinstance(value, torch.Tensor):
        if value.numel() > 0:
            print(f"  {key:20s}: {str(value.shape):25s} {value.dtype}")
        else:
            print(f"  {key:20s}: {'(empty)':25s}")
    elif isinstance(value, dict):
        print(f"  {key:20s}: dict ({len(value)} keys)")
    elif isinstance(value, list):
        print(f"  {key:20s}: list ({len(value)} items)")

print("\n" + "="*80)
print("✅ Ready for training!")
print("="*80)


Batch Structure:
  flux_image          : torch.Size([16, 1, 256, 256]) torch.float32
  sun_position        : torch.Size([16, 2])       torch.float32
  motor_position      : torch.Size([16, 2])       torch.float32
  focal_spot          : torch.Size([16, 3])       torch.float32
  target_name         : list (16 items)
  calibration_id      : list (16 items)
  heliostat_id        : list (16 items)
  deflectometry       : (empty)                  
  properties          : dict (7 keys)

✅ Ready for training!


In [6]:
# ============================================================================
# Training Loop Template
# ============================================================================

"""
for epoch in range(num_epochs):
    model.train()
    for batch in train_loader:
        flux = batch['flux_image']          # [B, 3, H, W]
        sun = batch['sun_position']         # [B, 2]
        target = batch['motor_position']    # [B, 2]
        
        pred = model(flux, sun)
        loss = criterion(pred, target)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Validation...
"""

"\nfor epoch in range(num_epochs):\n    model.train()\n    for batch in train_loader:\n        flux = batch['flux_image']          # [B, 3, H, W]\n        sun = batch['sun_position']         # [B, 2]\n        target = batch['motor_position']    # [B, 2]\n        \n        pred = model(flux, sun)\n        loss = criterion(pred, target)\n        \n        optimizer.zero_grad()\n        loss.backward()\n        optimizer.step()\n    \n    # Validation...\n"

In [4]:
    # Download deflectometry and properties
    print("\n📥 Step 2: Downloading deflectometry and properties...")
    client = StacClient(output_dir=str("./paint_data"))
    
    try:
        client.get_heliostat_data(heliostats=["AA31"], collections=['deflectometry'])
        print("✓ Deflectometry downloaded")
    except Exception as e:
        print(f"⚠️  Deflectometry: {e}")
    
    try:
        client.get_heliostat_data(heliostats=["AA31"], collections=['properties'])
        print("✓ Properties downloaded")
    except Exception as e:
        print(f"⚠️  Properties: {e}")


📥 Step 2: Downloading deflectometry and properties...
[2025-12-26 01:56:34,364][paint.data.stac_client][INFO] - Resuming download from checkpoint!
[2025-12-26 01:56:34,365][paint.data.stac_client][INFO] - All downloads complete. Checkpoint file removed.
✓ Deflectometry downloaded
[2025-12-26 01:56:34,367][paint.data.stac_client][INFO] - Loading catalogs for desired heliostats. 
[2025-12-26 01:56:34,640][paint.data.stac_client][INFO] - Processing heliostat catalog AA31-heliostat-catalog


Processing Items in Heliostat AA31-heliostat-catalog: 100%|██████████| 1/1 [00:00<00:00,  7.82Item/s]

✓ Properties downloaded
